# HAR 모형, 처음부터 다시

7번에서는 HAR을 빠르게 지나가고 HARQ로 넘어갔지. 이번엔 HAR만 천천히 볼게. 순서는 이래.

1. 무엇을 예측하려는 건가
2. 가장 단순한 예측 방법 세 가지와 각각의 문제
3. HAR의 아이디어
4. 숫자로 예측값 계산해보기
5. 계수는 어떻게 구하나
6. 계수가 말해주는 것
7. 왜 하필 1일·5일·22일인가
8. 네 프로젝트에서의 쓰임

---

## 1단계: 무엇을 예측하려는 건가

**내일의 변동성**이야. 수익률의 방향이 아니라 크기.

매일 그날의 변동성을 숫자 하나로 측정해뒀다고 하자. 앞에서 배운 RV(실현분산)일 수도 있고, 일봉만 있다면 고가·저가로 만든 레인지 변동성일 수도 있어. 이렇게 하루에 숫자 하나씩, 시계열이 쌓여 있어.

$$
RV_1,\ RV_2,\ \dots,\ RV_t \quad\longrightarrow\quad RV_{t+1} = \ ?
$$

오늘까지의 변동성 기록을 보고 내일 변동성을 맞히는 게 목표야.

---

## 2단계: 단순한 방법 세 가지와 각각의 문제

**방법 ① "내일은 오늘과 같다"**

$$
\widehat{RV}_{t+1} = RV_t
$$

변동성 군집(큰 날 다음엔 큰 날) 때문에 꽤 잘 맞아. 그런데 오늘 우연히 튄 날이었다면? 오늘 하루짜리 악재로 RV가 평소의 4배였어도 내일도 4배라고 예측해. **하루의 우연에 너무 휘둘려.**

**방법 ② "내일은 지난 한 달 평균과 같다"**

$$
\widehat{RV}_{t+1} = \frac{1}{22}\sum_{j=0}^{21} RV_{t-j}
$$

우연에는 강하지만, 어제 시장이 폭락했어도 한 달 평균에 1/22만 반영되니 **최근 변화에 너무 둔해.**

**방법 ③ AR(1) 회귀: "어제 값에 적당한 가중치를 줘서 예측"**

$$
RV_{t+1} = \beta_0 + \beta_1 RV_t + \varepsilon_{t+1}
$$

①과 ②의 중간쯤을 데이터가 알아서 정해줘. 그런데 이 모형에는 구조적인 문제가 있어. 오늘의 충격이 k일 뒤에 남는 비율이 $\beta_1^k$로 **지수적으로 빠르게** 사라져.

$\beta_1 = 0.7$이라면 이렇게 돼. 실제 변동성의 전형적인 모습과 나란히 놓아볼게(설명용 숫자야).

| 며칠 뒤 | AR(1)이 가정하는 영향 ($0.7^k$) | 실제 변동성의 전형적인 자기상관 |
|---|---|---|
| 1일 | 0.70 | 0.70 |
| 5일 | 0.17 | 0.60 |
| 22일 | 0.0004 | 0.45 |
| 60일 | 0 | 0.35 |

실제 변동성은 한 달, 두 달이 지나도 과거의 영향이 **꽤 남아 있어.** 폭락장의 불안이 몇 주, 몇 달씩 이어지는 걸 생각하면 자연스럽지. 이걸 **장기기억(long memory)**이라고 해. AR(1)은 이걸 전혀 못 잡아.

그렇다고 과거 60일을 전부 넣은 AR(60)을 쓰면 계수가 61개라 추정이 엉망이 돼.

---

## 3단계: HAR의 아이디어

해결책은 단순해. **과거를 세 덩어리로 요약해서 넣자.**

- **어제** 하루의 변동성
- **지난 1주일**(5 거래일)의 평균 변동성
- **지난 1달**(22 거래일)의 평균 변동성

$$
\boxed{RV_{t+1} = \beta_0 + \beta_d\,RV_t + \beta_w\,RV_t^{(w)} + \beta_m\,RV_t^{(m)} + \varepsilon_{t+1}}
$$

$$
RV_t^{(w)} = \frac{RV_t + RV_{t-1} + \dots + RV_{t-4}}{5}, \qquad RV_t^{(m)} = \frac{RV_t + RV_{t-1} + \dots + RV_{t-21}}{22}
$$

아래첨자 d, w, m은 daily, weekly, monthly야.

**2단계의 세 방법을 다 섞은 것**이라고 보면 돼. ①(어제), ②(한 달 평균), 그리고 그 중간(한 주 평균)에 각각 가중치를 주는데, **그 가중치를 데이터가 회귀로 정해주는 거야.**

**이름의 뜻 (Heterogeneous AutoRegressive):** 시장에는 **투자 기간이 서로 다른(heterogeneous)** 사람들이 섞여 있다는 직관에서 나왔어(Corsi 2009).

- 단타 트레이더는 **어제** 얼마나 흔들렸는지에 반응해.
- 스윙 트레이더는 **이번 주** 분위기에 반응해.
- 연기금이나 장기 기관은 **이번 달** 흐름에 반응해.

이 세 집단의 반응이 합쳐져서 내일의 변동성이 만들어진다는 거지. "AutoRegressive(자기회귀)"는 자기 자신의 과거 값으로 미래를 예측한다는 뜻이야.

---

## 4단계: 숫자로 예측해보기

계수가 이미 추정돼 있다고 하자(설명용 숫자야).

$$
\beta_0 = 0.1, \quad \beta_d = 0.4, \quad \beta_w = 0.35, \quad \beta_m = 0.2
$$

### 상황 A: 어제 갑자기 크게 흔들렸다

| | 값 |
|---|---|
| 어제 $RV_t$ | 4.0 |
| 지난주 평균 $RV_t^{(w)}$ | 2.5 |
| 지난달 평균 $RV_t^{(m)}$ | 1.5 |

$$
\widehat{RV}_{t+1} = 0.1 + 0.4(4.0) + 0.35(2.5) + 0.2(1.5) = 0.1 + 1.6 + 0.875 + 0.3 = \mathbf{2.875}
$$

| 방법 | 예측 |
|---|---|
| ① 어제와 같다 | 4.0 |
| ② 한 달 평균 | 1.5 |
| **HAR** | **2.875** |

HAR은 "어제의 급등이 **일부는 이어지겠지만 전부는 아닐 것**"이라고 판단해. 단순 방법들의 양 극단 사이에서, 과거 데이터가 알려준 비율만큼 섞은 거야.

### 상황 B: 한 달 내내 조용했다

어제, 지난주, 지난달이 모두 1.0이라면

$$
\widehat{RV}_{t+1} = 0.1 + 0.4 + 0.35 + 0.2 = \mathbf{1.05}
$$

조용했는데 예측이 오히려 **살짝 올라갔어.** 왜 그럴까?

### 장기 평균으로 돌아가려는 성질 (유도)

어제, 지난주, 지난달이 모두 같은 값 μ로 **영원히 유지되는** 상태를 생각해보자. 그러면 내일도 μ여야 하니까:

$$
\mu = \beta_0 + (\beta_d + \beta_w + \beta_m)\,\mu \quad\Rightarrow\quad \mu = \frac{\beta_0}{1 - (\beta_d + \beta_w + \beta_m)}
$$

$$
\mu = \frac{0.1}{1 - 0.95} = 2.0
$$

이 시장의 **장기 평균 변동성은 2.0**이야. 그래서

- 지금 변동성이 장기 평균보다 **낮으면**(상황 B, 1.0) → 예측이 **위로** 조금씩 올라가.
- 지금 변동성이 장기 평균보다 **높으면** → 예측이 **아래로** 조금씩 내려가.

변동성은 결국 평균으로 돌아온다는 성질(평균회귀)을 HAR이 자동으로 담고 있어. 세 계수의 합(0.95)이 1에 가까울수록 **돌아오는 속도가 느려**, 즉 과거의 영향이 오래 남아.

---

## 5단계: 계수는 어떻게 구하나

**그냥 OLS 회귀야.** 네가 패널 회귀에서 쓰던 것과 같아. 데이터를 이런 표로 만들어.

| 날짜 t | $RV_t$ (어제) | $RV_t^{(w)}$ (주 평균) | $RV_t^{(m)}$ (월 평균) | $RV_{t+1}$ (정답: 다음 날) |
|---|---|---|---|---|
| 2월 3일 | 1.2 | 1.4 | 1.6 | 1.1 |
| 2월 4일 | 1.1 | 1.3 | 1.6 | 2.8 |
| 2월 5일 | 2.8 | 1.6 | 1.7 | 2.2 |
| … | … | … | … | … |

- 한 행이 하루야.
- 왼쪽 세 열이 설명변수, 오른쪽 열이 종속변수야.
- 설명변수는 전부 **t일까지의 정보**로 만들고, 정답은 **t+1일**이야. 이 시점 구분이 16번의 look-ahead 문제를 막는 핵심이야.

이 표로 회귀를 돌리면 $\beta_0, \beta_d, \beta_w, \beta_m$이 나와. 예측할 때는 오늘 행의 세 값을 넣으면 돼.

세 변수가 서로 겹치는 날들로 만들어졌으니(월 평균 안에 주 평균의 날들이 들어 있음) 설명변수끼리 상관이 높아. 그래도 예측에는 문제가 없고, 개별 계수의 표준오차만 좀 커져. 표준오차는 7번에서 말했듯 Newey-West로 계산해.

---

## 6단계: 계수가 말해주는 것

추정된 계수 자체가 시장에 대한 정보야.

| 계수 패턴 | 의미 |
|---|---|
| $\beta_d$가 큼 | 단기 충격에 민감한 시장. 하루 변동이 다음 날까지 강하게 이어짐 |
| $\beta_m$이 큼 | 변동성이 천천히 움직이는 시장. 장기 흐름이 중요함 |
| 세 계수 합이 1에 가까움 | 변동성의 기억이 김. 한번 높아지면 오래 감 |

그래서 네 프로젝트에서 **"한국과 미국의 HAR 계수가 다른가?"**, **"평온 레짐과 혼란 레짐의 HAR 계수가 다른가?"** 같은 비교가 의미 있는 거야. 예를 들어 혼란 레짐에서 $\beta_d$가 커진다면, 위기 때는 어제의 충격이 내일로 훨씬 강하게 전달된다는 뜻이지.

---

## 7단계: 왜 하필 1일, 5일, 22일인가

솔직히 말하면 **이론적으로 정해진 숫자가 아니야.** 1주일이 5 거래일, 1달이 약 22 거래일이라는 달력 관행에서 나온 거야. 그런데 이 단순한 선택이 예측을 매우 잘해서 표준이 됐어.

핵심은 세 개의 평균을 섞으면 **과거 각 날짜에 대한 가중치가 계단 모양**이 된다는 거야. 4단계의 계수로 계산해보면:

| 과거 날짜 | 가중치 | 계산 |
|---|---|---|
| 어제 | 0.479 | $0.4 + 0.35/5 + 0.2/22$ |
| 2~5일 전 | 0.079 | $0.35/5 + 0.2/22$ |
| 6~22일 전 | 0.009 | $0.2/22$ |
| 23일 전 이후 | 0 | — |

**가까운 과거는 크게, 먼 과거도 조금은** 반영하는 구조야. 계수 4개로 2단계 표의 "천천히 사라지는 기억"을 흉내 내는 거지. AR(1)처럼 급격히 0이 되지 않아.

---

## 8단계: 네 프로젝트에서의 쓰임

**변동성 측정값으로 무엇을 넣나:** yfinance 일봉 기반이라면 고가·저가로 만든 레인지 변동성(Parkinson, Garman-Klass)을 RV 자리에 넣으면 돼. 수식은 그대로야.

**확장 모형들은 전부 HAR에 변수를 하나씩 더한 것:**

| 모형 | HAR에 추가한 것 | 묻는 질문 |
|---|---|---|
| **LHAR** | 과거 음의 수익률 $r^-_t$, $r^{-(w)}_t$, $r^{-(m)}_t$ | 하락이 미래 변동성을 더 키우나 |
| **SHAR** | 어제 RV를 $RS^+$, $RS^-$로 분리 | 하락 봉 변동성이 더 중요한가 |
| **HARQ** | $\sqrt{RQ_t} \times RV_t$ | 측정이 부정확한 날은 어제를 덜 믿어야 하나 |
| **레짐 HAR** | 레짐 더미와의 상호작용항 | 레짐마다 변동성 동학이 다른가 |

HAR을 이해하면 이 확장들은 **"변수 하나 추가한 회귀"**로 바로 읽혀.

**전략 S1의 엔진:** 변동성 타기팅은 HAR 예측값을 그대로 써.

$$
\text{주식 비중}_t = \min\left(\frac{\text{목표 변동성}}{\sqrt{\widehat{RV}_{t+1}}},\ 1\right)
$$

4단계 상황 A처럼 예측 변동성이 높아지면 비중을 줄이고, 조용하면 비중을 늘리는 거야. HAR이 잘 맞을수록 이 위험 조절이 정확해져.

**실무 팁:** 변동성은 가끔 극단적으로 튀어서 회귀를 한두 날이 좌우할 수 있어. 그래서 RV 대신 **로그 RV**나 **RV의 제곱근**(즉 변동성)으로 HAR을 돌리는 경우가 많아. 극단값의 영향이 줄어서 추정이 안정적이야. 기본형으로 먼저 해보고, 로그 버전과 표본 외 성과(QLIKE)를 비교해봐.

---

## 계산 코드

```python
import numpy as np
import pandas as pd
import statsmodels.api as sm
import yfinance as yf

# 일봉으로 Parkinson 변동성 (일별 분산 추정치)
px = yf.download('^KS11', start='2010-01-01', auto_adjust=False, progress=False)
rv = (np.log(px['High'] / px['Low']) ** 2 / (4 * np.log(2))).squeeze() * 1e4   # 단위: %²

d = pd.DataFrame({'RV': rv})
d['RV_w'] = d['RV'].rolling(5).mean()        # 지난 5일 평균 (오늘 포함)
d['RV_m'] = d['RV'].rolling(22).mean()       # 지난 22일 평균 (오늘 포함)
d['y'] = d['RV'].shift(-1)                   # 정답: 내일 RV
d = d.dropna()

X = sm.add_constant(d[['RV', 'RV_w', 'RV_m']])
har = sm.OLS(d['y'], X).fit(cov_type='HAC', cov_kwds={'maxlags': 22})
print(har.params)

b0, bd, bw, bm = har.params
print('장기 평균 변동성(분산):', b0 / (1 - bd - bw - bm))

# 오늘 기준 내일 예측
today = d.iloc[-1]
print('내일 예측 RV:', b0 + bd * today['RV'] + bw * today['RV_w'] + bm * today['RV_m'])
```

(이건 이해용으로 전체 표본을 한 번에 추정한 거야. 성과를 평가할 때는 12번의 워크포워드로, 매 시점 그때까지의 데이터로만 계수를 추정해야 해.)

---

**한 줄 요약:** HAR은 내일의 변동성을 **어제 하루 · 지난 1주 평균 · 지난 1달 평균** 세 개로 예측하는 회귀야. "어제와 같다"(너무 예민함)와 "한 달 평균과 같다"(너무 둔함)를 데이터가 정한 비율로 섞은 것이고, 이 단순한 구조로 변동성의 **긴 기억**과 **평균회귀**를 동시에 잡아내. 네 프로젝트의 LHAR, SHAR, HARQ, 레짐 HAR은 전부 이 식에 변수를 하나씩 더한 거야.

Question : LHAR, SHAR, HARQ, 레짐 HAR이 뭔데

---

# HAR의 네 가지 확장: LHAR, SHAR, HARQ, 레짐 HAR

네 모형 모두 **HAR 식에 무언가를 하나 더한 것**이야. 그래서 먼저 기본 HAR을 다시 적어두고, 각 확장이 **어떤 질문에 답하려고** 무엇을 더했는지를 하나씩 볼게.

$$
\text{HAR:}\quad RV_{t+1} = \beta_0 + \beta_d\,RV_t + \beta_w\,RV_t^{(w)} + \beta_m\,RV_t^{(m)} + \varepsilon_{t+1}
$$

아래 숫자 예시에서는 앞에서 쓴 기본 계수($\beta_0 = 0.1$, $\beta_d = 0.4$, $\beta_w = 0.35$, $\beta_m = 0.2$)를 그대로 쓰고, 새로 추가되는 계수만 설명용 숫자로 정할게. 실제로는 변수를 추가하면 기존 계수도 같이 바뀌지만, 차이를 보여주는 게 목적이라 단순화했어.

---

## 1. LHAR (Leverage HAR)

### 질문

**"같은 크기의 변동이라도, 가격이 떨어진 날이 오른 날보다 미래 변동성을 더 키우는가?"**

HAR은 어제 RV의 **크기**만 봐. RV는 제곱이라 부호가 없어서, −3% 폭락한 날과 +3% 급등한 날을 똑같이 취급해. 그런데 1장에서 말한 **레버리지 효과**(주가가 떨어지면 변동성이 커지는 현상) 때문에, 현실에서는 하락일 이후에 변동성이 더 커져.

### 식

HAR에 **과거 하락의 크기**를 변수로 추가해(Corsi & Renò 2012). 하락 크기를 이렇게 정의하자.

$$
L_t = \max(-r_t,\ 0)
$$

오늘 수익률 $r_t$가 −3%면 $L_t = 3$, +3%면 $L_t = 0$이야. 오른 날은 0이고, 떨어진 날만 그 크기가 기록돼.

$$
\boxed{RV_{t+1} = \underbrace{\beta_0 + \beta_d RV_t + \beta_w RV_t^{(w)} + \beta_m RV_t^{(m)}}_{\text{HAR 그대로}} + \underbrace{\gamma_d L_t + \gamma_w L_t^{(w)} + \gamma_m L_t^{(m)}}_{\text{추가: 하락의 크기}} + \varepsilon_{t+1}}
$$

$L_t^{(w)}$와 $L_t^{(m)}$은 RV와 똑같이 지난 5일, 22일 평균이야. 하락의 영향도 하루짜리, 주간, 월간으로 따로 보는 거지.

**기대 부호:** $\gamma > 0$. 하락이 클수록 미래 변동성이 커진다.

### 숫자 예시

어제 RV가 4.0으로 같은 두 날을 비교해보자. HAR 부분은 4번에서 계산한 대로 2.875야. $\gamma_d = 0.3$이라고 하자(주·월 하락 항은 편의상 생략).

| | 어제 수익률 | $L_t$ | 추가분 $0.3 \times L_t$ | LHAR 예측 |
|---|---|---|---|---|
| 폭락한 날 | −3% | 3 | 0.9 | **3.775** |
| 급등한 날 | +3% | 0 | 0 | **2.875** |

HAR이라면 두 날의 예측이 똑같이 2.875였을 거야. LHAR은 **폭락한 날 다음에 변동성이 더 클 것**이라고 구분해.

### 네 프로젝트에서의 위치

**일봉만 있으면 되는 모형이야.** 수익률의 부호는 일봉 종가로 바로 알 수 있으니까. 그래서 yfinance의 **A층(15년 이상 일봉)**에서 쓸 수 있는 반분산 계열 분석은 사실상 LHAR이야. **H1의 주력 모형**이 돼.

---

## 2. SHAR (Semivariance HAR)

### 질문

**"어제의 변동성 중에서 하락 봉에서 나온 부분이 상승 봉에서 나온 부분보다 미래 변동성을 더 키우는가?"**

LHAR과 비슷한 질문인데, 보는 해상도가 달라. LHAR은 하루 전체의 **종가 기준 부호 하나**만 봐. 그런데 하루 종가가 +0.5%여도 장중에 −3%까지 빠졌다가 회복했을 수 있지. SHAR은 **장중 봉 하나하나의 부호**를 봐.

### 식

HAR에서 어제 RV를 2번에서 배운 **상승 반분산과 하락 반분산으로 쪼개**(Patton & Sheppard 2015).

$$
RV_t = RS_t^+ + RS_t^- \quad\longrightarrow\quad \beta_d\,RV_t \ \text{을}\ \beta^+ RS_t^+ + \beta^- RS_t^- \ \text{로 교체}
$$

$$
\boxed{RV_{t+1} = \beta_0 + \underbrace{\beta^+ RS_t^+ + \beta^- RS_t^-}_{\text{어제 RV를 부호별로 분리}} + \beta_w RV_t^{(w)} + \beta_m RV_t^{(m)} + \varepsilon_{t+1}}
$$

**기대 결과:** $\beta^- > \beta^+$. 하락 봉에서 나온 변동성이 미래에 더 오래 영향을 준다.

**HAR과의 관계:** 만약 $\beta^+ = \beta^-$라면 $\beta^+ RS^+ + \beta^- RS^- = \beta(RS^+ + RS^-) = \beta RV$가 되니까 **그냥 HAR이야.** 그래서 SHAR의 핵심 검정은 **"$\beta^+ = \beta^-$인가?"**야. 같다면 쪼갤 필요가 없었다는 뜻이고, 다르다면 방향 정보가 중요하다는 뜻이지.

### 숫자 예시

어제 RV가 4.0으로 같은데 구성이 다른 두 날이야. $\beta^+ = 0.1$, $\beta^- = 0.7$이라고 하자. 주·월 부분은 $0.1 + 0.875 + 0.3 = 1.275$로 같아.

| | $RS^+$ | $RS^-$ | 어제 부분 $0.1 RS^+ + 0.7 RS^-$ | SHAR 예측 |
|---|---|---|---|---|
| 하락 봉 위주의 날 | 0.4 | 3.6 | $0.04 + 2.52 = 2.56$ | **3.835** |
| 상승 봉 위주의 날 | 3.6 | 0.4 | $0.36 + 0.28 = 0.64$ | **1.915** |

RV가 똑같은데 예측이 두 배 차이가 나. 2번 개념에서 "RV로는 급락일과 급등일을 구분 못 한다"고 했던 문제를 예측 모형 안에서 해결한 거야.

### 네 프로젝트에서의 위치

**장중 데이터가 필요해.** 일별 RS±를 만들려면 하루 안의 봉들이 있어야 하니까.

- **B층 (60분봉, 2년):** 하루 7개 봉이라 일별 RS±가 매우 부정확해(측정오차 약 53%). 가능은 하지만 결과를 조심해서 해석해야 해.
- **C층 (5분봉, 누적 수집):** 하루 78개 봉이라 제대로 된 SHAR이 가능해. 데이터가 쌓이는 대로 적용해.

그래서 역할 분담은 이렇게 돼. **A층 LHAR로 15년치 결론을 내고, C층 SHAR로 같은 결론이 장중 해상도에서도 나오는지 확인**하는 거야.

---

## 3. HARQ (HAR + Quarticity)

### 질문

**"어제 RV가 부정확하게 측정된 날에는, 어제 값을 덜 믿어야 하지 않나?"**

7번에서 자세히 다뤘으니 핵심만 다시 정리할게. RV는 봉 개수가 유한해서 **측정오차**가 있어. 그리고 그 오차는 **변동성이 큰 날일수록 훨씬 커져.** 그런데 HAR은 정확한 날이든 부정확한 날이든 어제 RV에 **항상 같은 가중치** $\beta_d$를 줘.

### 식

어제 RV의 가중치를 **그날의 측정 정확도에 따라** 바꿔. 측정 정확도는 7번의 realized quarticity($RQ_t = \frac{N}{3}\sum r_{t,i}^4$)로 추정해.

$$
\beta_{d,t} = \beta_d + \beta_{dQ}\,\sqrt{RQ_t}
$$

$$
\boxed{RV_{t+1} = \beta_0 + \underbrace{(\beta_d + \beta_{dQ}\sqrt{RQ_t})}_{\text{날마다 바뀌는 가중치}}RV_t + \beta_w RV_t^{(w)} + \beta_m RV_t^{(m)} + \varepsilon_{t+1}}
$$

회귀식으로 보면 설명변수 $\sqrt{RQ_t} \times RV_t$ **하나만 추가**한 거야.

**기대 부호:** $\beta_{dQ} < 0$. RQ가 큰 날(측정이 부정확한 날)에는 어제 RV의 가중치를 줄인다.

### 숫자 예시

$\beta_d = 0.5$, $\beta_{dQ} = -0.1$이라고 하자($\sqrt{RQ}$는 평균을 빼지 않은 원값으로 단순화).

| | $\sqrt{RQ_t}$ | 어제 RV의 실효 가중치 | 의미 |
|---|---|---|---|
| 조용한 날 | 1 | $0.5 - 0.1 = 0.4$ | 어제 값을 꽤 믿음 |
| 요동친 날 | 4 | $0.5 - 0.4 = 0.1$ | 어제 값을 거의 안 믿고 주·월 평균에 기댐 |

### 네 프로젝트에서의 위치

**RQ를 만들려면 장중 봉이 필요해.** 그래서 A층(일봉)에는 적용이 어렵고, B·C층에서만 쓸 수 있어. 솔직히 말하면 네 프로젝트에서 HARQ는 **우선순위가 낮아.** 주력 분석인 A층에서 쓸 수 없고, 연구 질문(방향 비대칭, 레짐)과도 직접 관련이 적어. C층 데이터가 충분히 쌓이면 "SHAR에 Q 조정을 더하면 더 좋아지나" 정도로 확인하면 충분해.

---

## 4. 레짐 HAR

### 질문

**"평온장과 혼란장에서 변동성이 움직이는 방식 자체가 다른가?"**

예를 들어 혼란장에서는 어제의 충격이 내일로 훨씬 강하게 전달될 수 있어. 공포가 공포를 부르니까. 그러면 $\beta_d$가 레짐마다 달라야 해. HAR은 모든 날에 같은 계수를 쓰니 이걸 못 잡아.

### 식: 두 가지 방법

**방법 (a) 관측된 레짐과의 상호작용 (추천)**

16번에서 만든 **online 레짐 더미** $D_t$(t일까지의 정보로 판단한 혼란장이면 1, 평온장이면 0)를 곱해서 넣어. 14번에서 본 "상호작용항 방식"이야.

$$
\boxed{RV_{t+1} = \underbrace{\beta_0 + \beta_d RV_t + \beta_w RV_t^{(w)} + \beta_m RV_t^{(m)}}_{\text{평온장의 HAR}} + \underbrace{D_t\big(\delta_0 + \delta_d RV_t + \delta_w RV_t^{(w)} + \delta_m RV_t^{(m)}\big)}_{\text{혼란장에서 달라지는 부분}} + \varepsilon_{t+1}}
$$

레짐별로 풀어 쓰면 이렇게 돼.

| 레짐 | 어제 RV의 계수 | 절편 |
|---|---|---|
| 평온 ($D_t = 0$) | $\beta_d$ | $\beta_0$ |
| 혼란 ($D_t = 1$) | $\beta_d + \delta_d$ | $\beta_0 + \delta_0$ |

**δ가 곧 "혼란장에서 얼마나 달라지나"**야. 그래서 $\delta_d = 0$인지 t-검정하는 게 연구 질문 "레짐별로 변동성 동학이 다른가"에 대한 직접적인 답이 돼.

**방법 (b) Markov switching HAR**

9번에서 본 방식이야. 레짐을 따로 정하지 않고, **레짐과 계수를 동시에 추정**해. 우아하지만 9번에서 경고했듯 "계수가 다른 구간"을 레짐으로 정의하니 **순환성**이 있어. 강건성 확인용으로만 써.

### 숫자 예시

$\beta_d = 0.3$, $\delta_d = 0.3$이라고 하자. 즉 평온장에서 어제 RV의 계수는 0.3이고 혼란장에서는 0.6이야. 어제 RV가 평소보다 2.0만큼 **더** 높았다면:

| 레짐 | 어제 충격이 내일 예측에 더하는 양 |
|---|---|
| 평온장 | $0.3 \times 2.0 = 0.6$ |
| 혼란장 | $0.6 \times 2.0 = 1.2$ |

같은 충격이 혼란장에서는 **두 배로** 내일에 전달돼.

### 네 프로젝트에서의 위치

레짐 더미는 지수 일봉으로 만드니까 **A층에서 바로 쓸 수 있어.** 그리고 LHAR과 결합하는 게 자연스러워.

$$
RV_{t+1} = \text{LHAR} + D_t \times (\text{LHAR의 각 항})
$$

이러면 **"혼란장에서 하락의 영향($\gamma$)이 더 커지는가"**까지 검정할 수 있어. 이게 네 프로젝트 **H1의 핵심 식**이야.

---

## 한눈에 비교

| 모형 | HAR에 더한 것 | 묻는 질문 | 필요 데이터 | 네 프로젝트 층 | 우선순위 |
|---|---|---|---|---|---|
| **LHAR** | 과거 하락 크기 $L_t$ | 하락일 이후 변동성이 더 커지나 | 일봉 | **A층 (15년+)** | **1순위** |
| **SHAR** | 어제 RV를 $RS^+$, $RS^-$로 분리 | 하락 봉 변동성이 더 중요한가 | 장중 봉 | C층 (B층은 보조) | 2순위 |
| **HARQ** | $\sqrt{RQ_t} \times RV_t$ | 부정확한 날은 어제를 덜 믿어야 하나 | 장중 봉 | C층 | 3순위 |
| **레짐 HAR** | 레짐 더미와의 상호작용 | 레짐마다 동학이 다른가 | 일봉 + 레짐 | **A층** | **1순위** |

**LHAR과 SHAR의 차이를 한 번 더 정리하면:**

| | LHAR | SHAR |
|---|---|---|
| 부호를 어디서 보나 | 하루 **종가** 수익률 하나 | 장중 **봉 하나하나** |
| 장중 급락 후 회복한 날 | 하락으로 안 잡힘 (종가가 +면 $L = 0$) | 하락 봉의 변동성이 $RS^-$에 잡힘 |
| 데이터 | 일봉이면 충분 | 장중 데이터 필요 |

같은 질문을 **해상도만 다르게** 묻는 거야.

---

## 모형 비교는 어떻게 하나

네 가지 모두 **표본 외 예측력**으로 HAR과 비교해야 해. 변수를 더하면 표본 내 적합도는 거의 항상 좋아지니까, 그건 증거가 안 돼.

1. 12번의 워크포워드로 매 시점 그때까지의 데이터로만 계수를 추정한다.
2. 다음 날 RV를 예측하고, 실제 값과 비교해 QLIKE와 MSE를 계산한다.
3. HAR 대비 손실이 유의하게 줄었는지 Diebold-Mariano 검정으로 확인한다.

---

## 계산 코드 (A층: 일봉만으로 LHAR + 레짐 HAR)

```python
import numpy as np
import pandas as pd
import statsmodels.api as sm
import yfinance as yf

px = yf.download('^KS11', start='2010-01-01', auto_adjust=False, progress=False)
px = px.droplevel(1, axis=1) if isinstance(px.columns, pd.MultiIndex) else px

d = pd.DataFrame(index=px.index)
d['RV'] = (np.log(px['High'] / px['Low']) ** 2 / (4 * np.log(2))) * 1e4   # Parkinson, %²
d['r'] = np.log(px['Close']).diff() * 100                                   # 일별 수익률, %
d['L'] = np.maximum(-d['r'], 0)                                             # 하락 크기

for v in ['RV', 'L']:
    d[f'{v}_w'] = d[v].rolling(5).mean()
    d[f'{v}_m'] = d[v].rolling(22).mean()
d['y'] = d['RV'].shift(-1)

# D: 16번 방식으로 만든 online 레짐 더미 (t일까지 정보), 여기서는 이미 있다고 가정
d['D'] = regime_online.reindex(d.index)

base = ['RV', 'RV_w', 'RV_m']
lev  = ['L', 'L_w', 'L_m']
for v in base + lev:
    d[f'{v}_xD'] = d[v] * d['D']
d = d.dropna()

def fit(cols):
    X = sm.add_constant(d[cols])
    return sm.OLS(d['y'], X).fit(cov_type='HAC', cov_kwds={'maxlags': 22})

har        = fit(base)
lhar       = fit(base + lev)
regime_lhar = fit(base + lev + ['D'] + [f'{v}_xD' for v in base + lev])

print(lhar.params[lev])                          # γ > 0 인가
print(regime_lhar.params.filter(like='_xD'))     # 혼란장에서 달라지는 부분 (δ)
```

(이해용으로 전체 표본을 한 번에 추정했어. 모형 비교는 워크포워드로 해야 해.)

---

**한 줄 요약:** 네 모형은 전부 HAR에 한 가지를 더한 거야. **LHAR**은 "하락한 날" 정보를, **SHAR**은 "하락한 장중 봉" 정보를, **HARQ**는 "측정 정확도"를, **레짐 HAR**은 "지금이 어떤 시장 국면인가"를 더했어. yfinance 일봉 기반인 네 프로젝트에서는 **LHAR + 레짐 HAR이 주력**이고, SHAR은 5분봉이 쌓이면 확인용으로, HARQ는 우선순위가 낮아.